In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Sirifort_Delhi_CPCB_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,352.0,138.0,241.0,112.0,176.0,244.0,106.0,55.0,87.0,129.0,347.0,301.0
1,2,366.0,NaN,135.0,130.0,161.0,161.0,131.0,67.0,83.0,155.0,332.0,304.0
2,3,361.0,156.0,133.0,140.0,235.0,160.0,116.0,72.0,83.0,158.0,373.0,284.0
3,4,385.0,158.0,158.0,149.0,256.0,242.0,48.0,81.0,65.0,185.0,370.0,200.0
4,5,335.0,137.0,145.0,170.0,242.0,241.0,77.0,74.0,59.0,151.0,376.0,189.0
5,6,344.0,147.0,154.0,138.0,208.0,180.0,56.0,69.0,68.0,150.0,251.0,182.0
6,7,350.0,137.0,174.0,152.0,266.0,269.0,56.0,70.0,62.0,145.0,372.0,236.0
7,8,375.0,138.0,176.0,143.0,187.0,223.0,53.0,52.0,74.0,160.0,391.0,311.0
8,9,386.0,169.0,186.0,143.0,148.0,151.0,90.0,48.0,91.0,146.0,359.0,173.0
9,10,298.0,199.0,199.0,167.0,155.0,161.0,106.0,47.0,101.0,142.0,336.0,266.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,352.000000,138.0,148.484848,112.000000,176.0,244.000000,106.0,55.000000,87.000000,129.0,347.000000,301.0
1,2,366.000000,152.1,135.000000,130.000000,161.0,161.000000,131.0,67.000000,83.000000,155.0,332.000000,304.0
2,3,361.000000,156.0,133.000000,140.000000,235.0,160.000000,116.0,72.000000,83.000000,158.0,373.000000,284.0
3,4,385.000000,158.0,158.000000,149.000000,256.0,242.000000,48.0,81.000000,65.000000,185.0,370.000000,200.0
4,5,335.000000,137.0,145.000000,170.000000,242.0,241.000000,77.0,74.000000,59.000000,151.0,376.000000,189.0
5,6,344.000000,147.0,154.000000,138.000000,208.0,180.000000,56.0,69.000000,68.000000,150.0,251.000000,182.0
6,7,350.000000,137.0,174.000000,152.000000,266.0,269.000000,56.0,70.000000,62.000000,145.0,372.000000,236.0
7,8,375.000000,138.0,176.000000,143.000000,187.0,223.000000,53.0,52.000000,74.000000,160.0,391.000000,311.0
8,9,386.000000,169.0,186.000000,143.000000,148.0,151.000000,90.0,48.000000,91.000000,146.0,359.000000,173.0
9,10,298.000000,199.0,199.000000,167.000000,155.0,161.000000,106.0,47.000000,101.000000,142.0,336.000000,266.0
